#  Run of Model **$X$** on the App **MASSA**
 
**Paramaters**
The parameters of all models should be available in the parent folder, in an EXCEL file.

### Imports

In [ ]:
import matplotlib.pyplot as plt
import pickle
from pathlib import Path
import torch

from neuralhydrology.evaluation import metrics
from neuralhydrology.nh_run import start_run, eval_run
from neuralhydrology.datasetzoo import register_dataset

### Adding our Dataset | Metrics | Models

In [ ]:
from custom.dataset.swisshourly import SwissHourly
register_dataset("swisshourly", SwissHourly)
from neuralhydrology.utils.config import Config

# Custom functions plot_training_curves
from custom.functions.Plot import plot_hydrograph, plot_training_curves, plot_validation_metrics
from custom.functions.utils_fla import parse_neuralhydrology_log


---

### Run parameters

In [ ]:
configPath = Path("M0.yml")

runFile = "M2_3"

cfg = Config(configPath)
TRAIN = True
Evaluation_TEST =  False


---
## **Training**

In [ ]:
import pandas as pd
csv_path = r"..\..\data\massa.csv"
data = pd.read_csv(csv_path)
print("Columns of the csv:  ", data.columns)
# print(data.dtypes)
# data['date'] = pd.to_datetime(data['date'])


# print some information about the dataset

# print(data.head())
# print(data.describe())

In [ ]:
if TRAIN:
    print("--- Starting the training of the model ---")
    # by default we assume that you have at least one cuda-capable nvidia gpu or macos with metal support
    if torch.cuda.is_available() or torch.backends.mps.is_available():
        print("Using GPU")
        start_run(config_file=Path(configPath))
    # fall back to cpu-only mode
    else:
        print("Using CPU")
        start_run(config_file=Path(configPath), gpu=-1)
    print("==================================")
    print("  Training ended successfully !   ")
    print("==================================")
else: print(" Training step skipped !")

2025-10-17 15:54:32,489: Epoch 3 average validation loss: 0.01195 -- Median validation metrics: avg_loss: 0.01195, MAPE: 23.11758, NSE: 0.91966, MSE: 44.49208, RMSE: 6.67024
2025-10-17 15:54:50,120: Epoch 4 average loss: avg_loss: 0.00365, avg_total_loss: 0.00365
2025-10-17 15:54:53,571: Epoch 4 average validation loss: 0.01856 -- Median validation metrics: avg_loss: 0.01856, MAPE: 35.38394, NSE: 0.91323, MSE: 48.05313, RMSE: 6.93204
2025-10-17 15:55:10,804: Epoch 5 average loss: avg_loss: 0.00427, avg_total_loss: 0.00427
2025-10-17 15:55:14,310: Epoch 5 average validation loss: 0.01172 -- Median validation metrics: avg_loss: 0.01172, MAPE: 22.94020, NSE: 0.94958, MSE: 27.92306, RMSE: 5.28423
2025-10-17 15:55:31,729: Epoch 6 average loss: avg_loss: 0.00273, avg_total_loss: 0.00273
2025-10-17 15:55:35,256: Epoch 6 average validation loss: 0.01101 -- Median validation metrics: avg_loss: 0.01101, MAPE: 21.49069, NSE: 0.95644, MSE: 24.12499, RMSE: 4.91172
2025-10-17 15:55:53,269: Epoch 7 a

### Plot of the Loss and Metrics

In [ ]:
# Path to your output.log file
log_path = Path(f"{cfg.base_run_dir}/{runFile}/output.log")
train_df, val_df = parse_neuralhydrology_log(log_path)

print("Plotting the losses and metrics throughout the training process: ")
plot_training_curves(train_df, val_df)
plot_validation_metrics(val_df)


---

## Evaluation on the **TEST** Dataset
The run directory that needs to be specified for evaluation is printed in the output log above. Since the folder name is created dynamically (including the date and time of the start of the run) you will need to change the `run_dir` argument according to your local directory name. By default, it will use the same device as during the training process.

In [ ]:
run_dir = Path(f"{cfg.base_run_dir}/{runFile}")

if Evaluation_TEST:
    print(" Starting the EVALUATION on the test period:  ")
    eval_run(run_dir=run_dir, period="test")
    print("==================================")
    print("  TEST ended successfully !   ")
    print("==================================")
else: print(" Testing process skipped !")

---

### Load and inspect model predictions
Next, we load the results file and compare the model predictions with observations. The results file is always a pickled dictionary with one key per basin (even for a single basin). The next-lower dictionary level is the temporal resolution of the predictions. 

In this case, we trained a model only on daily data ('1D'). Within the temporal resolution, the next-lower dictionary level are `xr`(an xarray Dataset that contains observations and predictions), as well as one key for each metric that was specified in the config file.

NOTE: The data variables in the xarray Dataset are named according to the name of the target variables, with suffix `_obs` for the observations and suffix `_sim` for the simulations.

In [ ]:
# model_str = f"model_epoch{str(cfg.epochs).zfill(3)}"
model_str = f"model_epoch040"
testPath = Path(run_dir / "test" / f"{model_str}" / "test_results.p")
with open(testPath, "rb") as fp:
    results = pickle.load(fp)
basin = "massa"
# results.keys()
results[basin]['1h']

### Plotting model predictions vs observations

Code for 1h prediction:

In [ ]:
# qobs=results['massa']['1h']['xr']['streamflow_obs']
# qsim=results['massa']['1h']['xr']['streamflow_sim']
# plot_hydrograph(
#     qobs=qobs,
#     qsim=qsim,
#     results=results,
#     basin='massa',
#     freq='1h',
#     plot_peaks=False  # turn on high-flow markers
# )

Code for 24h predictions:

In [ ]:
from custom.functions.Plot import plot_forecast_24h, plot_forecast_horizon_24h,plot_forecast_horizon_series

xr_ds = results[basin]["1h"]["xr"]
title24 = f"Massa – {cfg.predict_last_n}h | MAPE {results[basin]['1h']['MAPE']:.3f}, NSE {results[basin]['1h']['NSE']:.3f}, RMSE {results[basin]['1h']['MSE']:.3f}"

# last forecast of the horizon vs obs
plot_forecast_24h(xr_ds, title=title24).show()
# All spaghetti forecasts
# plot_forecast_24h(xr_ds, mode="spaghetti").show()

In [ ]:
# Example: visualize forecast issued on April 10, 2024, at midnight
plot_forecast_horizon_24h(
    xr_ds,
    issue_date="2024-08-10 00:00:00"
).show()
plot_forecast_horizon_24h(
    xr_ds,
    issue_date="2024-08-10 01:00:00"
).show()

In [ ]:
plot_forecast_horizon_series(
    xr_ds,
    issue_date="2024-06-10 00:00:00",
    n_forecasts=5
).show()

---

## Computation of all metric scores
Next, we are going to compute all metrics that are implemented in the NeuralHydrology package. You will find additional hydrological signatures implemented in `neuralhydrology.evaluation.signatures`.

In [ ]:
qobs=results[basin]['1h']['xr']['streamflow_obs']
qsim=results[basin]['1h']['xr']['streamflow_sim']
values = metrics.calculate_all_metrics(qobs.isel(time_step=-1), qsim.isel(time_step=-1))

for key, val in values.items():
    print(f"{key}: {val:.3f}")